# What this notebook is about

Public-private keypairs in notebook [Crypto - No Subject](https://drive.google.com/file/d/1INTQTi0bzOCs4U0hG2UJvc8ukZJb_vis/view?usp=sharing) were not associated with any identity. In practical applications of public key cryptography, public keys must be associated with *subjects* and such associations are described in *certificates*.

This notebook shows the creation of **self-signed certificates**, i.e., certificates where Subject=Issuer, **with an identity chosen arbitrarily**.

The main objective is convincing the reader that **anyone can create a self-signed certificate for anyone else**.

## Warning: password-protected keys

The commands below refer to files that contain private keys but are not protected by any password, for simplicity.

This is an insecure practice that should never be used in a production environment (unless you really know what you are doing, understand and accept the risk).

More details in "Protezione chiavi con password" in this page.

## File formats

The most widely used *certificate* format is X509.

Unfortunately, there are many different *file* formats for storing X509 certificates and the corresponding cryptographic keys.

The specific file format to use in a given application depends on the software that has to use the certificate (or the key).

The corresponding openssl commands are quite complex and intricate (a [small subset of them](https://www.xolphin.com/support/OpenSSL/Frequently_used_OpenSSL_Commands), just to have an idea).

See also "Crittografia in pratica - Qualche dettaglio" in [Reti di Calcolatori -
Approfondimenti Network Security](https://bartolialberto.github.io/ComputerNetworks/6%20-%20Network%20Security/Approfondimenti_Network_Security/).


Execute this command before executing the examples below.

In [ ]:
!cd ~/; openssl rand -writerand .rnd

# Generate a self-signed certificate

Generate a 1024 bit public-private keypair and a self-signed certificate (valid for 365 days, with RSA public key cryptography, 2048 bits key length) for a subject of your choice:

- private key in the file specified with `-keyout`
- certificate in the file specified with `-out`
- subject specified with `-subj`; if this option is not given, then openssl will ask the user interactively.

Note that you can specify the name of the subject without any constraint.

Thus, **anyone can create a self-signed certificate for anyone else**.


In [ ]:
!openssl req -x509 -nodes -days 365 -newkey rsa:2048 -keyout privateKey.key -out certificate.crt  -subj "/CN=www.unicredit.it"

Dump content of the certificate in textual form.

You will see that X509 certificates store a lot more information than (Subject, Public Key of Subject, Issuer, Expiration Date, Signature by Issuer). The usage of this information is beyond the scope of this notebook.

In [ ]:
!openssl x509 -in certificate.crt -noout -text

Have a look at the private key (stored in the default format of the openssl command used).

In [ ]:
!cat privateKey.key

## Play yourself



Try to generate a self-signed certificate with some "interesting" Subject.

The Subject fields that you can use are:
* CN: CommonName ("your" name or a DNS name)
* OU: OrganizationalUnit
* O: Organization
* L: Locality
* S: StateOrProvinceName
* C: CountryName
* emailAddress: email address

You need not specify all of them (the set of those that are required and of those that are optional depend on the software that will use the certificate, i.e., on the purpose of the certificate).

In [ ]:
!openssl req -x509 -nodes -days 365 -newkey rsa:2048 -keyout privateKey.key -out certificate.crt  -subj "PLACE WHAT YOU WANT HERE - LOOK AT ONE OF THE CODE CELLS ABOVE"

# HTTPS

The part below requires knowledge of HTTPS

##Download web server certificates

Fetch certificate from the HTTPS server at *google.com* and store it in file *googlecert.pem* in *pem* format

In [ ]:
! openssl s_client -connect google.com:443 -showcerts </dev/null | openssl x509 -outform pem > googlecert.pem

Dump pem file in text format.

You will see that a real certificate contains a lot more information than the simple tuple `(Subject, KPUB, Issuer)` considered in the lectures. Usage and meaning of such information is beyond the scope of this course. Just as an example, the Validity field specifies the time interval in which the certificate should be considered valid; the certificate verification procedure will check this constraint.


In [ ]:
!openssl x509 -in googlecert.pem -noout -text

## Is that Issuer in the KeySet/TrustSet ?

If you try to find the Issuer of the above certificate in the KeySet/TrustSet of your operating system or of your browser, probably you will not find it (you will not find a self-signed certificate for the Issuer in the KeySet/TrustSet of your operating system or browser).

The reason is beyond the scope of this notebook (and of this course). Just as a sort of "curiosity", in many practical cases certificates form a *chain*. The Issuer of the certificate is the Subject of *another* certificate that is *not* self-signed. The Issuer of this second certificate is the Subject of a *third* certificate that is *not* self-signed, and so on until finding a certificate that is self-signed. Only this last certificate must be stored in the KeySet/TrustSet of) your operating system or browser. This last certificate is called *trust anchor*.

In other words, in this course we consider only chains composed of two certificates: one that arrives from the outside (e.g., from an HTTPS server) and the other (the trust anchor) that is self-signed and that must be in the KeySet/TrustSet.

## Play yourself

Try to generate a self-signed certificate with a Subject identical to the one in the certificate of the HTTPS server of senato.it (or of any other "interesting" HTTPS server).

You need to:

1. Download a certificate from a website of your choice.
2. Dump the certificate content and take note of the Subject
3. Create a self-signed certificate for that Subject

The first code cell below corresponds to steps 1 and 2, the second code cell to step 3. Modify those cells as appropriate

In [ ]:
!openssl s_client -connect PLACE_WEB_SITE_NAME_HERE:443 -showcerts </dev/null | openssl x509 -outform pem > fetchedcert.pem
!openssl x509 -in fetchedcert.pem -noout -text

In [ ]:
!openssl req -x509 -nodes -days 365 -newkey rsa:2048 -keyout privateKey.key -out certificate.crt  -subj PLACE_YOUR_SUBJECT_HERE